# 06 — Neural text classification with PyTorch: embeddings, LSTM, CNN

This notebook is part of the NLP implementation pack for AI PMs, founders, and builders. It is designed to be runnable on toy data and easy to adapt to real company data.

## Mental model

Neural text classifiers learn embeddings and nonlinear patterns directly from tokens.

Pipeline:

```text
text → token ids → embedding layer → sequence encoder → classifier → softmax
```

This notebook builds small LSTM and CNN classifiers for sentiment. For production, compare them with TF-IDF baselines and Transformer models.

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

In [ ]:
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

df = pd.read_csv(DATA_DIR / "sample_customer_tickets.csv")
train_df, test_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["sentiment"])

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

label_encoder = LabelEncoder()
train_y = label_encoder.fit_transform(train_df["sentiment"])
test_y = label_encoder.transform(test_df["sentiment"])
label_encoder.classes_

In [ ]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Build vocabulary
word_to_id = {"<PAD>": 0, "<UNK>": 1}
for text in train_df["text"]:
    for tok in tokenize(text):
        word_to_id.setdefault(tok, len(word_to_id))

def encode(text):
    return torch.tensor([word_to_id.get(tok, 1) for tok in tokenize(text)], dtype=torch.long)

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.x = [encode(t) for t in texts]
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

def collate(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)
    xs = pad_sequence(xs, batch_first=True, padding_value=0)
    ys = torch.stack(ys)
    return xs, lengths, ys

train_loader = DataLoader(TextDataset(train_df["text"].tolist(), train_y), batch_size=8, shuffle=True, collate_fn=collate)
test_loader = DataLoader(TextDataset(test_df["text"].tolist(), test_y), batch_size=8, shuffle=False, collate_fn=collate)
print("Vocab size:", len(word_to_id))

## LSTM classifier

LSTMs read tokens sequentially and maintain hidden state. They are less parallel than Transformers but useful for learning sequence modeling basics.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, emb_dim=64, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        # concatenate last forward and backward hidden states
        features = torch.cat([h[-2], h[-1]], dim=1)
        return self.classifier(features)

model = LSTMClassifier(len(word_to_id), len(label_encoder.classes_))
model

In [ ]:
def train_model(model, loader, epochs=20, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for x, lengths, y in loader:
            logits = model(x, lengths)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += float(loss.detach())
        if (epoch + 1) % 5 == 0:
            print(f"epoch={epoch+1} loss={total/len(loader):.4f}")
    return model

def predict_model(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for x, lengths, y in loader:
            logits = model(x, lengths)
            preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.array(preds)

model = train_model(model, train_loader, epochs=5, lr=0.01)
preds = predict_model(model, test_loader)
print(classification_report(test_y, preds, target_names=label_encoder.classes_, zero_division=0))

## CNN text classifier

CNNs can capture local phrase patterns such as "charged twice", "app crashes", or "excellent support".

In [ ]:
class CNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, emb_dim=64, num_filters=64, kernel_sizes=(2,3,4)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, num_filters, k) for k in kernel_sizes])
        self.classifier = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x, lengths=None):
        emb = self.embedding(x).transpose(1, 2)  # batch, emb_dim, seq
        conv_outputs = []
        for conv in self.convs:
            if emb.size(2) < conv.kernel_size[0]:
                # pad short sequences
                pad_len = conv.kernel_size[0] - emb.size(2)
                emb_padded = nn.functional.pad(emb, (0, pad_len))
            else:
                emb_padded = emb
            h = torch.relu(conv(emb_padded))
            h = torch.max(h, dim=2).values
            conv_outputs.append(h)
        features = torch.cat(conv_outputs, dim=1)
        return self.classifier(features)

cnn = CNNTextClassifier(len(word_to_id), len(label_encoder.classes_))
cnn = train_model(cnn, train_loader, epochs=5, lr=0.01)
preds = predict_model(cnn, test_loader)
print(classification_report(test_y, preds, target_names=label_encoder.classes_, zero_division=0))

## Product note

Use neural classifiers when:

- you have enough labelled data
- word order matters
- you want to use pretrained embeddings or transfer learning

But always compare against TF-IDF. A cheaper baseline that works is often the better business decision.